# Lock v4, Step 1: estimation-pool diagnostics and lock emission

**dr_ensemble_v2 / partner-selection test. Protocol doc section 13.**

This notebook touches the **estimation** pool only. It:

1. loads the estimation/confirmation assignment and hard-asserts that no confirmation patient survives the filter;
2. computes ceiling-normalised complementarity `C` and weighted dependence `rho_cost` per pair, averaged over the nine cross-seed combinations;
3. runs the four pre-confirmation diagnostics (reliability; reliability by anchor strength; outcome granularity and tie structure; selection stability across training seeds);
4. resolves the primary endpoint (PDR, or grade 3 and worse) from diagnostic 3 using the threshold pre-committed in section 13;
5. writes `04_results/locked_config_v4.json` with file hashes, definitions, the resolved endpoint, the diagnostic results and the recorded power expectation.

**It does not read confirmation rows and does not compute the primary endpoint.** That happens once, in the confirmation notebook, after this lock file exists.

In [1]:
# Cell 1: environment, paths, configuration
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, itertools, datetime
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE    = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS  = f'{BASE}/01_splits'
PREDS   = f'{BASE}/03_predictions'
RESULTS = f'{BASE}/04_results'
AUDIT   = f'{BASE}/99_audit'
for d in (RESULTS, AUDIT): os.makedirs(d, exist_ok=True)

SPLIT_FILE = f'{SPLITS}/eyepacs_split_kaggletest_estconf_v2.csv'

# Locked analysis parameters (section 13)
BURDEN         = 0.25          # each model refers its own top 25%
REFERRAL_UNIT  = 'eye'         # 'eye' or 'patient'; see note in cell 4
W              = {4: 500, 3: 200, 2: 15}   # grade-1 miss penalty dropped (section 6)
W_FP           = 1
N_SPLITHALF    = 50
N_SUBSAMPLE    = 50
RNG_SEED       = 20260923

# Pre-committed fallback thresholds for diagnostic 3 (section 13)
TIE_MIN_DISTINCT = 6           # median distinct outcome values across 10 candidates
TIE_MAX_GROUP    = 3           # mean largest tie group

ROSTER = ['densenet121_binary','densenet121_5class',
          'efficientnet_b3_binary','efficientnet_b3_5class',
          'resnet50_binary','resnet50_5class',
          'vit_base_binary','vit_base_5class',
          'retfound_binary','retfound_5class','retfound_adversarial']
SEED_TAGS = ['', '_seed1', '_seed2']
PAIRS = list(itertools.combinations(ROSTER, 2))
assert len(ROSTER) == 11 and len(PAIRS) == 55

rng = np.random.default_rng(RNG_SEED)
print(f'{len(ROSTER)} architectures, {len(PAIRS)} pairs, {len(SEED_TAGS)} seeds')

Mounted at /content/drive
11 architectures, 55 pairs, 3 seeds


In [2]:
# Cell 2: load the estimation/confirmation assignment and assert pool separation

def pick_col(df, candidates, what):
    for c in candidates:
        if c in df.columns: return c
    raise AssertionError(f'Could not find the {what} column. Columns present: {list(df.columns)}')

split = pd.read_csv(SPLIT_FILE)
ID_COL    = pick_col(split, ['image_id','image','id','filename'], 'image id')
PAT_COL   = pick_col(split, ['patient_id','patient','pid','subject_id'], 'patient id')
GRADE_COL = pick_col(split, ['level','grade','true_grade','diagnosis'], 'grade')
POOL_COL  = pick_col(split, ['pool','split','set','assignment','estconf'], 'pool assignment')

pool_vals = set(split[POOL_COL].astype(str).str.lower().unique())
EST_KEY  = next((v for v in pool_vals if v.startswith('est')), None)
CONF_KEY = next((v for v in pool_vals if v.startswith('conf')), None)
assert EST_KEY and CONF_KEY, f'Unrecognised pool labels: {pool_vals}'
assert len(pool_vals) == 2, f'Expected exactly two pools, found: {pool_vals}'

pool_lc   = split[POOL_COL].astype(str).str.lower()
est_split = split[pool_lc == EST_KEY].copy()
CONF_PATIENTS = set(split.loc[pool_lc == CONF_KEY, PAT_COL].astype(str))
EST_PATIENTS  = set(est_split[PAT_COL].astype(str))

# Hard guardrails. Nothing downstream runs if these fail.
assert not (EST_PATIENTS & CONF_PATIENTS), 'Pools share patients: the split is not patient-disjoint.'
assert len(est_split[ID_COL]) == est_split[ID_COL].nunique(), 'Duplicate image ids in the estimation pool.'
CONF_IDS = set(split.loc[pool_lc == CONF_KEY, ID_COL].astype(str))
assert not (set(est_split[ID_COL].astype(str)) & CONF_IDS), 'Confirmation image ids leaked into the estimation frame.'

est_split = est_split.sort_values(ID_COL).reset_index(drop=True)
EST_IDS   = est_split[ID_COL].astype(str).to_numpy()
pat       = est_split[PAT_COL].astype(str).to_numpy()
grade     = est_split[GRADE_COL].to_numpy().astype(int)
referable = grade >= 2
n_eyes    = len(grade)

pat_codes, pat_index = np.unique(pat, return_inverse=True)
n_pat = len(pat_codes)

WMISS = np.zeros(n_eyes)
for g, w in W.items(): WMISS[grade == g] = w

print(f'Estimation pool: {n_eyes:,} eyes, {n_pat:,} patients')
print(f'  referable (grade>=2): {referable.sum():,} ({referable.mean():.4f})')
print(f'  grade 3: {(grade==3).sum():,}   grade 4 (PDR): {(grade==4).sum():,}   grade>=3: {(grade>=3).sum():,}')
print(f'Confirmation pool held back: {len(CONF_PATIENTS):,} patients, {len(CONF_IDS):,} eyes. Not loaded.')

Estimation pool: 35,718 eyes, 17,859 patients
  referable (grade>=2): 6,815 (0.1908)
  grade 3: 824   grade 4 (PDR): 824   grade>=3: 1,648
Confirmation pool held back: 8,929 patients, 17,858 eyes. Not loaded.


In [3]:
# Cell 3: load the 33 prediction files, hash them, restrict to the estimation pool

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''): h.update(blk)
    return h.hexdigest()

MODEL_KEYS = [f'{a}{t}' for a in ROSTER for t in SEED_TAGS]
assert len(MODEL_KEYS) == 33

P, FILE_SHA = {}, {}
for key in MODEL_KEYS:
    arch, tag = (key.rsplit('_seed', 1)[0], '_seed' + key.rsplit('_seed', 1)[1]) if '_seed' in key else (key, '')
    fn = f'eyepacs_predictions_kaggletest_{arch}{tag}_v2.csv'
    path = f'{PREDS}/{fn}'
    assert os.path.exists(path), f'Missing prediction file: {fn}'
    FILE_SHA[fn] = sha256_file(path)

    df = pd.read_csv(path)
    pid_col  = pick_col(df, ['image_id','image','id','filename'], 'image id')
    prob_col = pick_col(df, ['predicted_probability','prob_referable','probability','prob','p_referable'], 'probability')
    df[pid_col] = df[pid_col].astype(str)

    # Restrict to estimation, then assert the restriction actually happened.
    df = df[df[pid_col].isin(set(EST_IDS))]
    assert not (set(df[pid_col]) & CONF_IDS), f'{fn}: confirmation rows survived the estimation filter.'
    df = df.drop_duplicates(subset=pid_col).set_index(pid_col).reindex(EST_IDS)
    assert df[prob_col].notna().all(), f'{fn}: estimation eyes missing from predictions.'

    P[key] = df[prob_col].to_numpy(dtype=np.float64)

FILE_SHA[os.path.basename(SPLIT_FILE)] = sha256_file(SPLIT_FILE)
print(f'Loaded {len(P)} models x {n_eyes:,} estimation eyes at float64.')
print(f'Hashed {len(FILE_SHA)} files (33 predictions + 1 split assignment).')

Loaded 33 models x 35,718 estimation eyes at float64.
Hashed 34 files (33 predictions + 1 split assignment).


In [4]:
# Cell 4: core definitions
#
# Referral unit. REFERRAL_UNIT='eye' refers the top BURDEN fraction of eyes by score, which is
# the unit the severity-weighted cost is defined over. 'patient' ranks patients by their higher
# eye score and refers the top fraction, marking both eyes. Both induced rates are reported so
# the choice is visible rather than buried.

def refer_mask(scores, burden=BURDEN, sub=None, unit=REFERRAL_UNIT):
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    s = scores[idx]
    if unit == 'eye':
        k = int(round(burden * len(idx)))
        order = np.argsort(-s, kind='stable')
        m = np.zeros(len(idx), bool); m[order[:k]] = True
        return m
    codes, inv = np.unique(pat_index[idx], return_inverse=True)
    pmax = np.full(len(codes), -np.inf)
    np.maximum.at(pmax, inv, s)
    k = int(round(burden * len(codes)))
    order = np.argsort(-pmax, kind='stable')
    keep = np.zeros(len(codes), bool); keep[order[:k]] = True
    return keep[inv]

def m_miss(ref, sub_mask_w, sub_referable):
    '''Severity-weighted cost of missed referable eyes.'''
    return float(sub_mask_w[sub_referable & ~ref].sum())

def cost_contrib(ref, sub_mask_w, sub_referable):
    '''Per-case cost contribution: miss weight for a missed referable eye, W_FP for a false positive.'''
    c = np.zeros(len(ref))
    miss = sub_referable & ~ref
    c[miss] = sub_mask_w[miss]
    c[(~sub_referable) & ref] = W_FP
    return c

def pearson(a, b):
    if a.std() == 0 or b.std() == 0: return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def pair_quantities(a, b, sub=None):
    '''All per-pair quantities for architectures a and b, averaged over the nine seed combinations.'''
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    w, rf, gr = WMISS[idx], referable[idx], grade[idx]
    out = {k: [] for k in ('C','rho_cost','rho_binary','pdr_avoided','sev3_avoided',
                           'union_rate_eye','m_min','ceiling_pdr')}
    for ta in SEED_TAGS:
        ra = refer_mask(P[f'{a}{ta}'], sub=idx)
        for tb in SEED_TAGS:
            rb = refer_mask(P[f'{b}{tb}'], sub=idx)
            mA, mB = m_miss(ra, w, rf), m_miss(rb, w, rf)
            mboth  = float(w[rf & ~ra & ~rb].sum())
            mmin   = min(mA, mB)
            out['C'].append(1.0 - mboth / mmin if mmin > 0 else np.nan)
            out['m_min'].append(mmin)
            out['rho_cost'].append(pearson(cost_contrib(ra, w, rf), cost_contrib(rb, w, rf)))
            out['rho_binary'].append(pearson((rf != ra).astype(float), (rf != rb).astype(float)))
            for name, sel in (('pdr', gr == 4), ('sev3', gr >= 3)):
                miss_a, miss_b = int((sel & ~ra).sum()), int((sel & ~rb).sum())
                miss_both      = int((sel & ~ra & ~rb).sum())
                out[f'{name}_avoided'].append(min(miss_a, miss_b) - miss_both)
                if name == 'pdr': out['ceiling_pdr'].append(min(miss_a, miss_b))
            out['union_rate_eye'].append(float((ra | rb).mean()))
    return {k: float(np.nanmean(v)) for k, v in out.items()}

def marginal_cost(a, sub=None):
    '''Own weighted miss cost, averaged over seeds. Lower is better.'''
    idx = np.arange(n_eyes) if sub is None else np.asarray(sub)
    w, rf = WMISS[idx], referable[idx]
    return float(np.mean([m_miss(refer_mask(P[f'{a}{t}'], sub=idx), w, rf) for t in SEED_TAGS]))

print('Definitions loaded. C = 1 - m_both / min(m_A, m_B); union rule at each model\'s own 25% threshold.')

Definitions loaded. C = 1 - m_both / min(m_A, m_B); union rule at each model's own 25% threshold.


In [5]:
# Cell 5: per-pair quantities on the full estimation pool

rows = []
for a, b in PAIRS:
    q = pair_quantities(a, b)
    rows.append({'model_a': a, 'model_b': b, **q})
pairs_df = pd.DataFrame(rows)

MARGINAL = {a: marginal_cost(a) for a in ROSTER}
pairs_df['marginal_a'] = pairs_df.model_a.map(MARGINAL)
pairs_df['marginal_b'] = pairs_df.model_b.map(MARGINAL)

pairs_df.to_csv(f'{RESULTS}/pair_quantities_estimation_v4.csv', index=False)

solo_rate_eye = {a: float(np.mean([refer_mask(P[f'{a}{t}']).mean() for t in SEED_TAGS])) for a in ROSTER}
solo_rate_pat = {a: float(np.mean([
    len(np.unique(pat_index[refer_mask(P[f'{a}{t}'])])) / n_pat for t in SEED_TAGS])) for a in ROSTER}

print(pairs_df[['model_a','model_b','C','rho_cost','pdr_avoided','sev3_avoided','union_rate_eye']]
      .sort_values('C', ascending=False).head(10).to_string(index=False))
print(f'\nC: mean {pairs_df.C.mean():.4f}, range {pairs_df.C.min():.4f} to {pairs_df.C.max():.4f}')
print(f'rho_cost: mean {pairs_df.rho_cost.mean():.4f}')
print(f'PDR avoided (union, own-25%): mean {pairs_df.pdr_avoided.mean():.3f}, '
      f'range {pairs_df.pdr_avoided.min():.3f} to {pairs_df.pdr_avoided.max():.3f}')
print(f'grade>=3 avoided: mean {pairs_df.sev3_avoided.mean():.3f}')
print(f'\nSingle-model referral rate at BURDEN={BURDEN}: '
      f'{np.mean(list(solo_rate_eye.values())):.4f} of eyes, '
      f'{np.mean(list(solo_rate_pat.values())):.4f} of patients')
print(f'Union referral rate: {pairs_df.union_rate_eye.mean():.4f} of eyes '
      f'(range {pairs_df.union_rate_eye.min():.4f} to {pairs_df.union_rate_eye.max():.4f})')

               model_a                model_b        C  rho_cost  pdr_avoided  sev3_avoided  union_rate_eye
    densenet121_5class        retfound_5class 0.485295  0.351292    22.444444     47.666667        0.332627
    densenet121_5class   retfound_adversarial 0.478742  0.372800    23.444444     47.888889        0.333352
    densenet121_5class        retfound_binary 0.472371  0.376486    20.777778     46.555556        0.332630
efficientnet_b3_5class        resnet50_5class 0.468117  0.336758    36.444444     62.000000        0.330562
    densenet121_binary   retfound_adversarial 0.463085  0.375126    22.222222     45.333333        0.332500
    densenet121_binary        retfound_5class 0.462288  0.364837    20.666667     43.444444        0.332123
efficientnet_b3_5class   retfound_adversarial 0.457091  0.349839    23.888889     48.555556        0.336587
    densenet121_binary        retfound_binary 0.451948  0.383757    19.111111     42.555556        0.331806
efficientnet_b3_binary   ret

In [6]:
# Cell 6: diagnostics 1 and 2. Reliability of C and rho_cost, overall and by anchor strength.

STRONG = [a for a in ROSTER if MARGINAL[a] <= np.median(list(MARGINAL.values()))]
WEAK   = [a for a in ROSTER if a not in STRONG]

def halves(seed):
    r = np.random.default_rng(seed)
    perm = r.permutation(n_pat)
    h1 = set(perm[: n_pat // 2].tolist())
    m1 = np.isin(pat_index, list(h1))
    return np.where(m1)[0], np.where(~m1)[0]

rel = {k: [] for k in ('C','rho_cost','C_strong','C_weak')}
for rep in range(N_SPLITHALF):
    i1, i2 = halves(RNG_SEED + rep)
    q1 = {(a, b): pair_quantities(a, b, sub=i1) for a, b in PAIRS}
    q2 = {(a, b): pair_quantities(a, b, sub=i2) for a, b in PAIRS}
    for stat in ('C','rho_cost'):
        v1 = [q1[p][stat] for p in PAIRS]; v2 = [q2[p][stat] for p in PAIRS]
        rel[stat].append(spearmanr(v1, v2, nan_policy='omit').statistic)
    for grp, label in ((STRONG,'C_strong'), (WEAK,'C_weak')):
        sub = [p for p in PAIRS if p[0] in grp or p[1] in grp]
        rel[label].append(spearmanr([q1[p]['C'] for p in sub],
                                    [q2[p]['C'] for p in sub], nan_policy='omit').statistic)

def spearman_brown(r_half, k=2.0):
    return float(k * r_half / (1 + (k - 1) * r_half))

# Spearman-Brown k is expressed relative to one half of the estimation pool
K_EST  = 2.0
K_CONF = len(CONF_PATIENTS) / (n_pat / 2)

reliability = {}
for k, v in rel.items():
    rh = float(np.nanmean(v))
    reliability[k] = {'split_half': round(rh, 4),
                      'full_estimation_pool': round(spearman_brown(rh, K_EST), 4),
                      'confirmation_pool_projected': round(spearman_brown(rh, K_CONF), 4)}

for k, v in reliability.items():
    print(f'{k:10s}  split-half {v["split_half"]:.3f}  '
          f'full estimation {v["full_estimation_pool"]:.3f}  '
          f'confirmation projected {v["confirmation_pool_projected"]:.3f}')

R_EST  = reliability['C']['full_estimation_pool']
R_CONF = reliability['C']['confirmation_pool_projected']
GATE   = R_EST >= 0.80
print(f'\nStrong anchors: {STRONG}')
print(f'Weak anchors:   {WEAK}')
print(f'\nReliability gate (>=0.80 on the full estimation pool): {"PASS" if GATE else "FAIL"} at {R_EST:.3f}')
print('  PASS: a null speaks to the mechanism.  FAIL: a null is a power statement and the implied n is reported.')

C           split-half 0.589  full estimation 0.741  confirmation projected 0.589
rho_cost    split-half 0.494  full estimation 0.661  confirmation projected 0.494
C_strong    split-half 0.601  full estimation 0.751  confirmation projected 0.601
C_weak      split-half 0.419  full estimation 0.591  confirmation projected 0.419

Strong anchors: ['densenet121_binary', 'densenet121_5class', 'vit_base_binary', 'retfound_binary', 'retfound_5class', 'retfound_adversarial']
Weak anchors:   ['efficientnet_b3_binary', 'efficientnet_b3_5class', 'resnet50_binary', 'resnet50_5class', 'vit_base_5class']

Reliability gate (>=0.80 on the full estimation pool): FAIL at 0.741
  PASS: a null speaks to the mechanism.  FAIL: a null is a power statement and the implied n is reported.


In [7]:
# Cell 7: diagnostic 3. Outcome granularity and tie structure, at confirmation granularity.
# Measured on confirmation-sized random patient subsamples of the estimation pool, because the
# question is whether the outcome can be ranked at the granularity confirmation will have.

N_CONF_PAT = len(CONF_PATIENTS)
gran = {'pdr': {'distinct': [], 'maxtie': [], 'spread': []},
        'sev3': {'distinct': [], 'maxtie': [], 'spread': []},
        'pdr_seed0': {'distinct': [], 'maxtie': [], 'spread': []}}

def tie_profile(values):
    v = np.round(np.asarray(values, dtype=float), 6)
    _, counts = np.unique(v, return_counts=True)
    return len(counts), int(counts.max()), float(np.nanmax(v) - np.nanmin(v))

for rep in range(N_SUBSAMPLE):
    r = np.random.default_rng(RNG_SEED + 10_000 + rep)
    keep = set(r.choice(n_pat, size=N_CONF_PAT, replace=False).tolist())
    sub = np.where(np.isin(pat_index, list(keep)))[0]

    q = {(a, b): pair_quantities(a, b, sub=sub) for a, b in PAIRS}
    # seed-0-only integer version, the pessimistic granularity bound
    w, rf, gr = WMISS[sub], referable[sub], grade[sub]
    q0 = {}
    for a, b in PAIRS:
        ra, rb = refer_mask(P[a], sub=sub), refer_mask(P[b], sub=sub)
        sel = gr == 4
        q0[(a, b)] = min(int((sel & ~ra).sum()), int((sel & ~rb).sum())) - int((sel & ~ra & ~rb).sum())

    for anchor in ROSTER:
        cand = [p for p in PAIRS if anchor in p]
        assert len(cand) == 10
        for key, vals in (('pdr',       [q[p]['pdr_avoided'] for p in cand]),
                          ('sev3',      [q[p]['sev3_avoided'] for p in cand]),
                          ('pdr_seed0', [q0[p] for p in cand])):
            d, t, s = tie_profile(vals)
            gran[key]['distinct'].append(d); gran[key]['maxtie'].append(t); gran[key]['spread'].append(s)

granularity = {}
for key, v in gran.items():
    granularity[key] = {'median_distinct_of_10': round(float(np.median(v['distinct'])), 2),
                        'mean_max_tie_group':    round(float(np.mean(v['maxtie'])), 2),
                        'mean_within_anchor_spread': round(float(np.mean(v['spread'])), 3)}
    print(f'{key:10s}  median distinct {granularity[key]["median_distinct_of_10"]:.2f} of 10  '
          f'mean largest tie group {granularity[key]["mean_max_tie_group"]:.2f}  '
          f'mean spread {granularity[key]["mean_within_anchor_spread"]:.3f}')

pdr_ok = (granularity['pdr']['median_distinct_of_10'] >= TIE_MIN_DISTINCT and
          granularity['pdr']['mean_max_tie_group']    <= TIE_MAX_GROUP)
PRIMARY_ENDPOINT = 'pdr_misses_avoided' if pdr_ok else 'severe3plus_misses_avoided'
ENDPOINT_DECIDED_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat()

print(f'\nPre-committed thresholds: median distinct >= {TIE_MIN_DISTINCT}, mean largest tie group <= {TIE_MAX_GROUP}')
print(f'PDR outcome rankable: {pdr_ok}')
print(f'PRIMARY ENDPOINT = {PRIMARY_ENDPOINT}')
if not pdr_ok:
    print('  Fallback triggered. PDR becomes the key secondary, per section 13.')
print(f'Decided at {ENDPOINT_DECIDED_UTC}, before any confirmation row was read.')

pdr         median distinct 10.00 of 10  mean largest tie group 1.50  mean spread 8.271
sev3        median distinct 10.00 of 10  mean largest tie group 1.42  mean spread 13.256
pdr_seed0   median distinct 7.00 of 10  mean largest tie group 2.67  mean spread 10.518

Pre-committed thresholds: median distinct >= 6, mean largest tie group <= 3
PDR outcome rankable: True
PRIMARY ENDPOINT = pdr_misses_avoided
Decided at 2026-09-23T12:21:34.384734+00:00, before any confirmation row was read.


In [8]:
# Cell 8: diagnostic 4. Selection stability across training seeds.
# If a rule's top pick changes with the training draw, selection fails on training variance
# alone, independently of measurement.

def picks_for_seed(tag):
    w, rf = WMISS, referable
    R = {a: refer_mask(P[f'{a}{tag}']) for a in ROSTER}
    marg = {a: m_miss(R[a], w, rf) for a in ROSTER}
    out = {}
    for anchor in ROSTER:
        cand = [c for c in ROSTER if c != anchor]
        stats = {}
        for c in cand:
            mA, mB = marg[anchor], marg[c]
            mboth = float(w[rf & ~R[anchor] & ~R[c]].sum())
            mmin = min(mA, mB)
            stats[c] = {'C': 1.0 - mboth / mmin if mmin > 0 else np.nan,
                        'rho_cost': pearson(cost_contrib(R[anchor], w, rf), cost_contrib(R[c], w, rf)),
                        'marginal': mB}
        out[anchor] = {
            'least_weighted_dependence': min(cand, key=lambda c: stats[c]['rho_cost']),
            'least_ceiling_overlap':     max(cand, key=lambda c: stats[c]['C']),
            'best_partner_marginal':     min(cand, key=lambda c: stats[c]['marginal'])}
    return out

seed_picks = {t if t else '_seed0': picks_for_seed(t) for t in SEED_TAGS}
RULES = ['least_weighted_dependence','least_ceiling_overlap','best_partner_marginal']

seed_stability = {}
tags = list(seed_picks)
for rule in RULES:
    agree = []
    for anchor in ROSTER:
        p = [seed_picks[t][anchor][rule] for t in tags]
        agree.append(np.mean([p[i] == p[j] for i in range(len(p)) for j in range(i + 1, len(p))]))
    seed_stability[rule] = {'mean_pairwise_agreement': round(float(np.mean(agree)), 3),
                            'anchors_unanimous': int(sum(a == 1.0 for a in agree)),
                            'chance': round(1 / 10, 3)}
    print(f'{rule:28s} mean pairwise agreement {seed_stability[rule]["mean_pairwise_agreement"]:.3f}  '
          f'unanimous on {seed_stability[rule]["anchors_unanimous"]}/11 anchors  (chance 0.100)')

least_weighted_dependence    mean pairwise agreement 0.212  unanimous on 1/11 anchors  (chance 0.100)
least_ceiling_overlap        mean pairwise agreement 0.212  unanimous on 0/11 anchors  (chance 0.100)
best_partner_marginal        mean pairwise agreement 0.000  unanimous on 0/11 anchors  (chance 0.100)


In [9]:
# Cell 9: write locked_config_v4.json and the diagnostics file

LOCKED_AT = datetime.datetime.now(datetime.timezone.utc).isoformat()

power_expectation = {
    'note': 'Recorded before the confirmation pool is opened so a null is pre-interpreted, not explained afterwards.',
    'expected_estimation_pool_reliability': R_EST,
    'expected_confirmation_pool_reliability': R_CONF,
    'attenuation_factor_sqrt_product': round(float(np.sqrt(max(R_EST, 0) * max(R_CONF, 0))), 3),
    'v3_observed_weighted_dependence_correlation': 0.18,
    'v3_disattenuated_underlying_estimate': 0.33,
    'expected_observed_correlation_here': round(0.33 * float(np.sqrt(max(R_EST, 0) * max(R_CONF, 0))), 3),
    'detection_threshold_eleven_anchors': 0.28,
    'interpretation': ('Expected effect and detection threshold coincide. A null does not rule out a '
                       'relationship of the size v3 implies; only a substantially stronger one registers.'),
    'implied_requirements': {'patients_for_reliability_0.90': 25000,
                             'architectures_for_threshold_0.20': 21,
                             'patient_axis_satisfied': True,
                             'architecture_axis_satisfied': False}}

lock = {
    'lock': 'v4',
    'locked_at_utc': LOCKED_AT,
    'locked_on': ('Kaggle-test estimation pool only (%d patients, %d eyes). '
                  'Confirmation pool (%d patients) not read by this notebook.'
                  % (n_pat, n_eyes, len(CONF_PATIENTS))),
    'protocol_doc_section': 13,
    'file_sha256': FILE_SHA,
    'definitions': {
        'complementarity_C': 'C = 1 - m_both / min(m_A, m_B), severity-weighted, ceiling-normalised',
        'weighted_dependence': 'rho_cost = Pearson correlation of per-case cost contributions',
        'miss_weights': {str(k): v for k, v in W.items()},
        'false_positive_weight': W_FP,
        'grade_1_miss_penalty': 'dropped (section 6)',
        'burden': BURDEN,
        'referral_unit': REFERRAL_UNIT,
        'primary_aggregation': "union rule at each model's own 25% threshold",
        'secondary_aggregation': 'max-score aggregation at matched referral volume',
        'seed_handling': 'all pair quantities averaged over the nine cross-seed combinations',
        'algebraic_link': ('Within a sample, union advantage = min(m_A, m_B) x C. The predictor is one '
                           'factor of the outcome; the test is out of sample, which is what makes it a test.')},
    'competing_rules': {
        'least_weighted_dependence': 'minimise rho_cost (estimation)',
        'least_ceiling_overlap': 'maximise C (estimation)',
        'best_partner_marginal': "minimise partner's own weighted miss cost (estimation)",
        'random': 'uniform over the ten candidates, 1,000 draws',
        'oracle': 'maximise the confirmation outcome; upper bound only'},
    'primary_endpoint': {
        'statistic': ('mean across 11 anchors of the within-anchor Spearman correlation between '
                      'estimation-measured C and the confirmation outcome across that anchor 10 candidates'),
        'outcome': PRIMARY_ENDPOINT,
        'outcome_decided_at_utc': ENDPOINT_DECIDED_UTC,
        'outcome_decision_basis': 'diagnostic 3 granularity thresholds pre-committed in section 13',
        'uncertainty': ('bootstrap over architectures, 1000 resamples, anchors and candidate pools rebuilt '
                        'from the resampled multiset, self-pairs excluded, repeated architectures contributing '
                        'as distinct anchor instances'),
        'supported_if': 'point estimate positive and the 95 percent percentile interval excludes zero',
        'also_reported': ('anchor permutation interval (not the decision rule: pairs recur across anchors so '
                          'the eleven within-anchor correlations are not exchangeable); partial correlation '
                          'controlling for estimation-measured partner marginal quality')},
    'decision_relevant_secondary': ('per-anchor paired difference in the confirmation outcome between '
                                    'least_ceiling_overlap and best_partner_marginal, architecture-bootstrapped'),
    'reliability_gate': {'threshold': 0.80, 'observed_full_estimation_pool': R_EST, 'passed': bool(GATE),
                         'if_passed': 'a null speaks to the mechanism',
                         'if_failed': 'a null is a power statement and the implied sample size is reported'},
    'power_expectation': power_expectation,
    'named_limitation': ('The union primary is not volume-controlled. Holding each model to its own 25% means '
                         'union volume varies only through overlap in the referred sets, so a low-overlap partner '
                         'delivers both more complementarity and more referrals. The matched-volume max-score '
                         'secondary separates them. A positive primary with a null secondary means selection works '
                         'through volume, which is weaker than complementarity but still true, and is reported so.'),
    'exploratory_no_verdict': ['10 to 40 percent burden curve',
                               'severity-weighted index as a consistency check against the count endpoint',
                               'max-score matched-volume results alongside the union rule',
                               'top-1 policy simulation in the v3 format for comparability',
                               'results with and without retfound_adversarial'],
    'confirmation_status': 'CLOSED. Opened once, by the confirmation notebook, after this file exists.'}

diagnostics = {
    'generated_utc': LOCKED_AT,
    'pool': {'estimation_patients': n_pat, 'estimation_eyes': n_eyes,
             'confirmation_patients': len(CONF_PATIENTS), 'confirmation_eyes': len(CONF_IDS),
             'grade_counts': {str(g): int((grade == g).sum()) for g in range(5)},
             'referable_rate': round(float(referable.mean()), 4)},
    'referral_rates': {'single_model_eye': round(float(np.mean(list(solo_rate_eye.values()))), 4),
                       'single_model_patient': round(float(np.mean(list(solo_rate_pat.values()))), 4),
                       'union_eye_mean': round(float(pairs_df.union_rate_eye.mean()), 4)},
    'diagnostic_1_reliability': reliability,
    'diagnostic_2_anchor_strength': {'strong_anchors': STRONG, 'weak_anchors': WEAK,
                                     'C_strong': reliability['C_strong'], 'C_weak': reliability['C_weak']},
    'diagnostic_3_granularity': granularity,
    'diagnostic_4_seed_stability': seed_stability,
    'pair_summary': {'C_mean': round(float(pairs_df.C.mean()), 4),
                     'rho_cost_mean': round(float(pairs_df.rho_cost.mean()), 4),
                     'pdr_avoided_mean': round(float(pairs_df.pdr_avoided.mean()), 3),
                     'sev3_avoided_mean': round(float(pairs_df.sev3_avoided.mean()), 3)}}

with open(f'{RESULTS}/locked_config_v4.json', 'w') as f: json.dump(lock, f, indent=2)
with open(f'{RESULTS}/estimation_diagnostics_v4.json', 'w') as f: json.dump(diagnostics, f, indent=2)

lock_sha = hashlib.sha256(json.dumps(lock, indent=2).encode()).hexdigest()
print(f'Wrote locked_config_v4.json  sha256 {lock_sha}')
print(f'Wrote estimation_diagnostics_v4.json')
print(f'Wrote pair_quantities_estimation_v4.csv')
print(f'\nLOCK v4 IS WRITTEN. Primary endpoint: {PRIMARY_ENDPOINT}. Confirmation may now be opened once.')

Wrote locked_config_v4.json  sha256 9b78257ece485d0266b222a69e48f8eaa94f7b351911238d16195a5b087eb6cc
Wrote estimation_diagnostics_v4.json
Wrote pair_quantities_estimation_v4.csv

LOCK v4 IS WRITTEN. Primary endpoint: pdr_misses_avoided. Confirmation may now be opened once.


In [10]:
# Cell 10: append this run to the provenance record

entry = f'''

## Lock v4 estimation run ({LOCKED_AT[:10]})

Kaggle-test portion: 53,576 images, 26,788 patients, labels from Kaggle discussion 16149, preprocessed
with the recovered v1 pipeline. Pixel-level non-overlap verified across all 88,702 images by SHA-256 over
decoded pixels and a 256-bit perceptual hash; an earlier 2,243-match figure came from a 64-bit hash and is
superseded.

Split `01_splits/eyepacs_split_kaggletest_estconf_v2.csv` (seed 20260923, patient-disjoint, label-free at
assignment): estimation {n_pat:,} patients / {n_eyes:,} eyes, confirmation {len(CONF_PATIENTS):,} patients /
{len(CONF_IDS):,} eyes. Inference complete for all 33 models on the Kaggle-test portion
(`eyepacs_predictions_kaggletest_*_v2.csv`).

Produced by `99_audit/ESTIMATION_lock_v4_diagnostics.ipynb`. Outputs: `04_results/locked_config_v4.json`
(sha256 {lock_sha[:16]}...), `04_results/estimation_diagnostics_v4.json`,
`04_results/pair_quantities_estimation_v4.csv`.

Diagnostics: C reliability {R_EST:.3f} on the full estimation pool (gate 0.80, {"pass" if GATE else "fail"});
C reliability strong anchors {reliability['C_strong']['full_estimation_pool']:.3f} versus weak
{reliability['C_weak']['full_estimation_pool']:.3f}; primary endpoint resolved to {PRIMARY_ENDPOINT} from the
granularity diagnostic; seed stability of the top pick reported per rule against a chance level of 0.100.

The confirmation pool was not read. It is opened once, by the confirmation notebook.
'''

with open(f'{AUDIT}/RUN_LOG.md', 'a') as f: f.write(entry)
print('Appended to 99_audit/RUN_LOG.md')
print(entry)

Appended to 99_audit/RUN_LOG.md


## Lock v4 estimation run (2026-09-23)

Kaggle-test portion: 53,576 images, 26,788 patients, labels from Kaggle discussion 16149, preprocessed
with the recovered v1 pipeline. Pixel-level non-overlap verified across all 88,702 images by SHA-256 over
decoded pixels and a 256-bit perceptual hash; an earlier 2,243-match figure came from a 64-bit hash and is
superseded.

Split `01_splits/eyepacs_split_kaggletest_estconf_v2.csv` (seed 20260923, patient-disjoint, label-free at
assignment): estimation 17,859 patients / 35,718 eyes, confirmation 8,929 patients /
17,858 eyes. Inference complete for all 33 models on the Kaggle-test portion
(`eyepacs_predictions_kaggletest_*_v2.csv`).

Produced by `99_audit/ESTIMATION_lock_v4_diagnostics.ipynb`. Outputs: `04_results/locked_config_v4.json`
(sha256 9b78257ece485d02...), `04_results/estimation_diagnostics_v4.json`,
`04_results/pair_quantities_estimation_v4.csv`.

Diagnostics: C reliability 0.741 on the full estimatio

## After this notebook

1. Read `04_results/estimation_diagnostics_v4.json`. Four things decide whether the confirmation run is worth making: the reliability gate, the strong-versus-weak reliability gap, the resolved primary endpoint, and seed stability. If seed stability sits at chance, selection has already failed on training variance and that is the finding.
2. Move this executed notebook to `99_audit/ESTIMATION_lock_v4_diagnostics.ipynb`.
3. Only then the confirmation notebook, which reads the confirmation pool once, computes the primary and the decision-relevant secondary under the locked configuration, and writes `04_results/confirmation_results_v4.json`.